In [0]:
catalog = "firstdeproject"
schema = "silver"

tables = [t.tableName for t in spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect() if not t.isTemporary]

for t in tables:
    print(f"\n=== {catalog}.{schema}.{t} ===")
    spark.table(f"{catalog}.{schema}.{t}").show(5, truncate=False)

In [0]:
# Create star schema (dim_customers, dim_products, fact_sales)

from pyspark.sql import functions as F, Window

CATALOG, SILVER, GOLD = "firstdeproject", "silver", "gold"
META = ["_source_folder", "_source_file", "_ingested_at"]

def silver(t):
    return spark.table(f"{CATALOG}.{SILVER}.{t}")

def save_gold(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable(f"{CATALOG}.{GOLD}.{name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD}")


In [0]:
## 3. fact_sales
# Grain: one row per order line (sls_ord_num + product). sls_ord_num stays in the fact as a degenerate dimension.

# COMMAND ----------
sd = silver("sales_details").drop(*META)
dc = spark.table(f"{CATALOG}.{GOLD}.dim_customers").select("customer_key", "customer_id")
dp = spark.table(f"{CATALOG}.{GOLD}.dim_products").select("product_key", "product_number")

fact_sales = (
    sd.join(dp, sd.sls_prd_key == dp.product_number, "left")
      .join(dc, sd.sls_cust_id.cast("string") == dc.customer_id, "left")
      .select(
          F.col("sls_ord_num").alias("order_number"),
          "product_key",
          "customer_key",
          F.col("sls_order_dt").alias("order_date"),
          F.col("sls_ship_dt").alias("shipping_date"),
          F.col("sls_due_dt").alias("due_date"),
          F.col("sls_sales").alias("sales_amount"),
          F.col("sls_quantity").alias("quantity"),
          F.col("sls_price").alias("price"),
      ))

save_gold(fact_sales, "fact_sales")

## 4. Integrity checks (orphans = fact rows with no matching dimension row)


f = spark.table(f"{CATALOG}.{GOLD}.fact_sales")
print("fact rows:", f.count())
print("orphan customers:", f.filter("customer_key IS NULL").count())
print("orphan products :", f.filter("product_key IS NULL").count())

In [0]:
fact_sales.show(5)